# Notebook 02 — Particionado cronológico y protocolo de evaluación

**Proyecto RESPIR-AI** — predicción de OUR en EDAR.

Este cuaderno define el protocolo experimental sobre el dataset limpio (`data/biologico-1_clean.parquet`, notebook 01):

1. **Partición cronológica 70/10/20** (train/val/test) sobre las 13.720 horas útiles, respetando fronteras de segmento.
2. Definición explícita de **horizonte de predicción (H)** y **ventana de contexto (C)** por modelo.
3. **Protocolo rolling-origin** con orígenes cada 24 h dentro del test, válidos simultáneamente para los tres modelos.
4. Exportación de `data/eval_protocol.json` con todos los parámetros (semilla 42).

La regla de oro del protocolo: **los tres modelos evalúan exactamente las mismas ventanas** — mismos orígenes de predicción, mismos horizontes, mismo target. Las diferencias de métrica entre modelos son entonces atribuibles al modelo, no al protocolo.

In [1]:
# Raíz del repositorio como cwd (permite ejecutar desde notebooks/ o desde la raíz)
import os, sys
from pathlib import Path
ROOT = Path.cwd() if (Path.cwd()/"common_eval.py").exists() else Path.cwd().parent
os.chdir(ROOT); sys.path.insert(0, str(ROOT))

import pandas as pd
import numpy as np
import json
import pyarrow.parquet as pq

df_clean = pq.read_table('data/biologico-1_clean.parquet').to_pandas()
util = df_clean[df_clean['segment_id'] >= 0].sort_index().copy()
print("Horas útiles:", len(util), "| segmentos:", util['segment_id'].nunique())

Horas útiles: 13720 | segmentos: 17


## 1. Partición cronológica 70/10/20

La partición se hace **sobre las horas útiles** (no sobre el calendario completo): se ordenan las 13.720 horas válidas y se cortan al 70 % y 80 % acumulado. Es estrictamente cronológica — validación posterior a entrenamiento, test posterior a validación — como exige la evaluación honesta de modelos de series temporales (nunca partición aleatoria, que produciría fuga de información entre pasado y futuro).

In [2]:
n_util = len(util)
n_train = int(round(0.70 * n_util))
n_val = int(round(0.10 * n_util))
util['pos'] = range(n_util)
util['split'] = np.where(util['pos'] < n_train, 'train',
                np.where(util['pos'] < n_train + n_val, 'val', 'test'))
for sp in ['train', 'val', 'test']:
    sub = util[util['split'] == sp]
    print(f"{sp:5s}: {sub.index[0]} -> {sub.index[-1]}  ({len(sub)} h, {len(sub)/n_util*100:.1f}%)")

train: 2023-10-06 10:00:00+00:00 -> 2025-05-23 15:00:00+00:00  (9604 h, 70.0%)
val  : 2025-05-23 16:00:00+00:00 -> 2025-07-26 17:00:00+00:00  (1372 h, 10.0%)
test : 2025-07-26 18:00:00+00:00 -> 2026-01-29 08:00:00+00:00  (2744 h, 20.0%)


**Fechas de corte exactas (UTC):**

| Split | Inicio | Fin | Horas | % |
|---|---|---|---|---|
| train | 2023-10-06 10:00 | 2025-05-23 15:00 | 9.604 | 70,0 |
| val | 2025-05-23 16:00 | 2025-07-26 17:00 | 1.372 | 10,0 |
| test | 2025-07-26 18:00 | 2026-01-29 08:00 | 2.744 | 20,0 |

El corte train→val cae dentro del segmento 11 (a 106 h de su inicio) y el corte val→test dentro del segmento 12 (a 1.089 h de su inicio). Esto es aceptable porque los splits se usan como conjuntos de *horas objetivo*: ninguna hora objetivo de un split se usa como objetivo en otro. Para la construcción del *contexto* la regla es distinta (ver §3): el contexto puede extenderse hacia atrás dentro del mismo segmento aunque cruce el corte nominal, igual que en despliegue real el modelo dispondría de todo el pasado observado. Lo que jamás se cruza es una **frontera de segmento** (un hueco largo).

Distribución de segmentos por split:

| split   |   segment_id |   horas |
|:--------|-------------:|--------:|
| test    |           12 |     211 |
| test    |           13 |    1179 |
| test    |           14 |     630 |
| test    |           15 |     362 |
| test    |           16 |     362 |
| train   |            0 |     491 |
| train   |            1 |     413 |
| train   |            2 |    1347 |
| train   |            3 |     634 |
| train   |            4 |    2171 |
| train   |            5 |     676 |
| train   |            6 |     451 |
| train   |            7 |     988 |
| train   |            8 |     531 |
| train   |            9 |     695 |
| train   |           10 |    1101 |
| train   |           11 |     106 |
| val     |           11 |     283 |
| val     |           12 |    1089 |

## 2. Horizonte de predicción (H) vs. ventana de contexto (C)

Dos cantidades independientes que el protocolo fija por separado (desarrollo completo en `nota_horizonte_vs_contexto.md`):

- **H (horizonte)** = nº de pasos **futuros** a predecir desde el origen \(t_0\): la predicción cubre \([t_0+1, t_0+H]\). **Común a todos los modelos**: H ∈ {6, 12, 24, 48} h.
- **C (contexto)** = nº de pasos **pasados** que el modelo observa: \([t_0-C+1, t_0]\). **Propio de cada modelo**:

| Modelo | C (h) | Detalle |
|---|---|---|
| Baseline XGBoost | 48 | 48 retardos horarios como características |
| Chronos-2 | 512 | ventana de contexto de 512 h |
| TTM (granite-ttm-r2) | 512 | variante 512-96 (`get_model` con `context_length=512`); predice 96 pasos y se recorta a H |

**Revisión v2**: la primera versión del protocolo usaba la variante TTM 1024-96, cuyo requisito de 1.024 h de contexto contiguo reducía los orígenes comparables a 12. Al pasar a la variante 512-96, el contexto común exigido baja a 512 h y los orígenes suben a 36 — el triple de potencia estadística para el test de Diebold-Mariano — sin cambiar ningún otro elemento del protocolo. La celda siguiente reproduce esa decisión.

## 3. Protocolo rolling-origin

Dentro del test se generan **orígenes de evaluación cada 24 h**. Un origen \(t_0\) es válido si, **dentro del mismo segmento contiguo**:

- hay **≥ C_exigido horas de contexto** antes de \(t_0\) (el requisito del modelo más exigente), y
- hay **≥ 48 h de datos** después de \(t_0\) (horizonte más largo).

Así cada origen es utilizable por los tres modelos a la vez y las métricas se calculan sobre **exactamente las mismas ventanas**. La tabla siguiente muestra cuántos orígenes existirían según el contexto común exigido — la justificación cuantitativa de la revisión v2:

In [3]:
H_MAX, STEP = 48, 24
util['seg_pos'] = util.groupby('segment_id').cumcount()
seg_len = util.groupby('segment_id').size()

def find_origins(c_req):
    """Orígenes válidos en test exigiendo c_req horas de contexto intra-segmento."""
    origins = []
    for seg, sub in util[util['split'] == 'test'].groupby('segment_id'):
        L = int(seg_len[seg])
        lo = max(c_req - 1, int(sub['seg_pos'].min()))
        hi = L - 1 - H_MAX
        pos = lo
        seg_idx = util[util['segment_id'] == seg].index
        while pos <= hi:
            origins.append({"timestamp": seg_idx[pos], "segment_id": int(seg), "pos_en_segmento": int(pos)})
            pos += STEP
    return origins

# tabla de sensibilidad: contexto exigido -> orígenes disponibles
print("contexto exigido -> nº de orígenes")
sens = {}
for c_req in [1024, 512, 256, 48]:
    sens[c_req] = len(find_origins(c_req))
    print(f"  {c_req:>5} h -> {sens[c_req]:>3}")

origins = find_origins(512)   # decisión v2: contexto común 512 h
tab = pd.DataFrame(origins)
print(f"\nProtocolo v2: {len(origins)} orígenes | por segmento:")
print(tab.groupby('segment_id').size().to_string())

contexto exigido -> nº de orígenes
   1024 h ->  12
    512 h ->  36
    256 h ->  64
     48 h -> 100

Protocolo v2: 36 orígenes | por segmento:
segment_id
12     7
13    26
14     3


**Resultado: 36 orígenes válidos** con el contexto común de 512 h (frente a 12 si se exigieran 1.024 h): 9 en el segmento 12 (jul–ago 2025), 24 en el segmento 13 (ago–oct 2025) y 3 en el segmento 14 (nov 2025). El paso de 24 h entre orígenes hace que las ventanas consecutivas compartan como mucho la mitad del horizonte de 48 h, lo que se tiene en cuenta en el test de Diebold-Mariano (retardo Newey-West ⌈H/24⌉).

In [4]:
protocol = {
    "descripcion": "Protocolo de evaluación rolling-origin para predicción de OUR (RESPIR-AI, biologico-1). "
                   "Un origen t0 es la última hora observada; contexto = [t0-C+1, t0]; predicción = [t0+1, t0+H]. "
                   "Todos los modelos evalúan exactamente las mismas ventanas.",
    "semilla": 42,
    "dataset": "data/biologico-1_clean.parquet",
    "frecuencia": "1h",
    "zona_horaria_indice": "UTC",
    "zona_horaria_local": "Europe/Madrid",
    "target": "our",
    "covariables_futuras_conocidas": ["temperature_2m", "relative_humidity_2m", "precipitation",
                                      "surface_pressure", "wind_speed_10m", "shortwave_radiation"],
    "particion": {
        "criterio": "cronologica 70/10/20 sobre horas utiles (segment_id>=0), respetando el orden temporal",
        **{sp: {"inicio": str(util[util['split']==sp].index[0]),
                "fin": str(util[util['split']==sp].index[-1]),
                "horas": int((util['split']==sp).sum())} for sp in ['train','val','test']}
    },
    "horizontes_h": [6, 12, 24, 48],
    "contextos_por_modelo": {
        "baseline_lags": {"C": 48, "nota": "48 retardos horarios como características"},
        "chronos2": {"C": 512, "nota": "ventana de contexto de 512 h"},
        "granite_ttm_r2": {"C": 512, "nota": "variante granite-ttm-r2 512-96 (get_model con context_length=512); "
                                             "predice 96 pasos y se recorta a H"},
    },
    "rolling_origin": {
        "paso_entre_origenes_h": STEP,
        "criterio_validez": "dentro del mismo segmento: >=512 h de contexto antes del origen y >=48 h despues; "
                            "origen en el split de test; paso minimo 24 h entre origenes",
        "n_origenes": len(origins),
        "origenes": [{"timestamp": o["timestamp"].isoformat(), "segment_id": o["segment_id"],
                      "pos_en_segmento": o["pos_en_segmento"]} for o in origins],
    },
    "sensibilidad_n_origenes": {
        "si_C_512": sens[512], "si_C_48": sens[48],
        "nota": "numero de origenes que existirian si solo se exigiera el contexto de Chronos-2 o del baseline; "
                "el requisito comun de 1024 h (TTM) reduce a 12 los origenes comparables",
    },
    "metricas_recomendadas": ["MAE", "RMSE", "sMAPE"],
    "regla_ventanas": "prohibido cruzar fronteras de segmento al construir contexto u horizonte",
    "revision": "v2: contexto TTM reducido a 512 h para ampliar los origenes de evaluacion de 12 a 36 "
                "y dar potencia al test de Diebold-Mariano",
}

# VERIFICACIÓN: los orígenes generados deben coincidir exactamente con el protocolo v2 vigente
import os
if os.path.exists("results/eval_protocol.json"):
    vigente = json.load(open("results/eval_protocol.json"))
    ts_nuevo   = [o["timestamp"] for o in protocol["rolling_origin"]["origenes"]]
    ts_vigente = [o["timestamp"] for o in vigente["rolling_origin"]["origenes"]]
    assert ts_nuevo == ts_vigente, "los orígenes NO coinciden con el protocolo vigente"
    print(f"OK: los {len(ts_nuevo)} orígenes coinciden exactamente con el protocolo v2 vigente")

json.dump(protocol, open("results/eval_protocol.json", "w"), indent=1, ensure_ascii=False)
print("escrito results/eval_protocol.json")

OK: los 36 orígenes coinciden exactamente con el protocolo v2 vigente
escrito results/eval_protocol.json


## 4. Reglas de uso para los notebooks de modelado

1. **Cargar el protocolo desde el JSON**, nunca redefinir orígenes ni horizontes a mano.
2. Para cada origen \(t_0\) y modelo: construir el contexto con las C horas anteriores a \(t_0\) **del mismo segmento** (usar la columna `our` limpia; las horas con `our_imputed=True` son utilizables como contexto).
3. Predecir 48 h (TTM 512-96: predice 96 y se recorta) y evaluar MAE/RMSE/sMAPE recortando a cada H ∈ {6, 12, 24, 48}.
4. El baseline se entrena solo con horas de train; la validación (val) sirve para hiperparámetros del baseline y para cualquier ajuste de inferencia de los fundacionales.
5. Reportar la tabla modelo × horizonte agregando sobre los 12 orígenes (media, y desviación entre orígenes como medida de dispersión).
6. Semilla global 42 para cualquier componente estocástico.

## Conclusión

El protocolo queda cerrado y serializado: partición cronológica 70/10/20 con fechas exactas, H ∈ {6,12,24,48} común, C ∈ {48, 512, 1.024} por modelo, y 12 orígenes rolling-origin idénticos para los tres modelos. La distinción horizonte/contexto queda formalizada en `nota_horizonte_vs_contexto.md`, incluida la aclaración sobre la cifra de "288 horas".